# EP03｜向量與矩陣：內積、矩陣乘法、線性變換

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep03/ep03_vectors_matrices.ipynb)

這份 notebook 對應網站上的 [EP03 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep03/)，段落和網站一樣，可以邊讀邊跑：

1. 工具一：向量（含加法與純量乘法的圖）
2. 工具二：內積與餘弦相似度（含「長度 × 影子」和餘弦定理的圖）
3. 工具三：矩陣與矩陣乘法
4. 工具四：線性變換
5. **本集重點：內積＝相似度，矩陣乘法＝一次算很多個內積**
6. 動手做：從零寫出 `dot`、`norm`、`cosine_similarity`、`matvec`、`matmul` 並用測試檢查；寫出畫變形圖的 `draw`；和 NumPy 比速度
7. 弄壞它：5 個「程式照跑、答案卻不對」或直接報錯的例子

最後是參考解答，以及一格選做：載入 Qwen2.5-0.5B，看真實嵌入的相似度表。

這份 notebook 只用到 NumPy 和 matplotlib，Colab 已經內建，不用安裝任何東西；EP01 寫好的 `softmax` 也已經定義在下面，不用匯入。在自己的電腦上，用 EP01 建立的 `llm-labs` 專案開啟即可。動手做的最後一個檢查會和 PyTorch 比對：Colab 已經內建 PyTorch；沒裝 PyTorch 的話（EP01 步驟 7），那個檢查會自動跳過。

**使用方式**：從上到下依序執行（Shift + Enter）。標著「先猜猜看」的地方，先在心裡猜結果再執行。參考解答在最後，先自己寫寫看。

In [ ]:
# Colab 已經內建 NumPy 和 matplotlib，不用安裝；在自己的電腦上，請用 EP01 建立的 llm-labs 環境。
# %pip install -q numpy matplotlib

import math
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)

這份 notebook 用 NumPy 2.5 測試過。程式格裡的註解是說明，印出來的結果就在格子下方。圖上的文字用英文，因為 matplotlib 預設的字型沒有中文，會變成方框。

**寫法約定**：向量寫成 $(3, 1)$，形狀寫成 `(2,)`。向量用小寫（$a$、$x$），矩陣用大寫（$A$、$W$）。$a_i$ 是 $a$ 的第 $i$ 個數字；數學從 1 開始數，Python 從 0 開始數。

下面是 EP01 動手做寫好的 `softmax`（數值穩定版），本集重點會用到它：

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

In [ ]:
def softmax(z, axis=-1):
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和

## 1. 工具一：向量

### 一排數字＝箭頭＝點

**向量 vector** 就是一排數字，例如 $a = (3, 1)$。同一個向量有三種看法：**一排數字**（程式裡的一維陣列，形狀 `(2,)`）、**箭頭**（從原點 $(0, 0)$ 指到 $(3, 1)$，有方向也有長度）、**點**（箭頭的尖端）。2 個數字畫在平面上（2D），3 個數字畫在空間裡（3D）。

> **注意**：「維」有兩種意思。EP01 說的陣列維度 dimension 是有幾個軸（`ndim`）；向量的維度是有幾個數字。所以 `(896,)` 是一維陣列，也是 896 維向量。

### 加法與純量乘法

- **加法**：對應位置相加，$(3, 1) + (1, 2) = (4, 3)$。畫成箭頭是「頭尾相接」：把 $b$ 的尾巴接到 $a$ 的頭，從原點到 $b$ 的新尖端就是 $a + b$。
- **純量乘法 scalar multiplication**：**純量 scalar** 是單獨一個數。純量乘向量，就是每個數字都乘上它：$2a$ 方向不變、長度變 2 倍；$-a$ 指向反方向；$0.5a$ 縮短一半。

下面這張圖的左邊是頭尾相接（灰色虛線是平移到 $a$ 尖端的 $b$），右邊是 $2a$、$a$、$0.5a$、$-a$，全部在同一條直線上。畫箭頭用 `ax.annotate("", xy=終點, xytext=起點, arrowprops=...)`：第一個參數是要寫的字，這裡留空，只畫箭頭。

In [ ]:
def arrow(ax, start, end, color="black", ls="-"):
    """從 start 畫一支箭頭到 end。"""
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="-|>", color=color, linestyle=ls, linewidth=1.5))


a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.4))

# 左：頭尾相接
arrow(ax1, (0, 0), a)
arrow(ax1, (0, 0), b)
arrow(ax1, a, a + b, color="gray", ls="--")     # 把 b 的尾巴接到 a 的頭
arrow(ax1, (0, 0), a + b, color="C0")
ax1.text(1.6, 0.25, "a")
ax1.text(0.15, 1.2, "b")
ax1.text(3.6, 1.7, "b", color="gray")
ax1.text(0.8, 2.7, "a + b = (4, 3)", color="C0")
ax1.set(xlim=(-0.5, 4.8), ylim=(-0.5, 3.5), aspect="equal", title="a + b")

# 右：純量乘法，全部在同一條直線上
for k, name, color, dy in [(2, "2a", "C0", 0.35), (1, "a", "black", 0.35), (0.5, "0.5a", "C2", -0.6), (-1, "-a", "C3", 0.35)]:
    arrow(ax2, (0, 0), k * a, color=color)
    ax2.text(*(k * a + np.array([-0.4, dy])), name, color=color)
ax2.set(xlim=(-3.8, 6.8), ylim=(-1.8, 2.8), aspect="equal", title="2a, a, 0.5a, -a")

for ax in (ax1, ax2):
    ax.grid(True)
plt.show()

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])
print(a + b)
print(a - b)
print(2 * a, -1 * a, 0.5 * a)

NumPy 的 `+`、`*` 本來就是逐元素運算（EP01），直接寫就好。

### 長度與單位向量

$$
\lVert a \rVert = \sqrt{\sum_{i=1}^{n} a_i^2} = \sqrt{a_1^2 + a_2^2 + \cdots + a_n^2}
$$

白話：每個數字先平方，全部加起來（EP02 的 $\sum$），再開根號。

這是向量的**長度**，也就是箭頭有多長。$\lVert a \rVert$ 念「a 的長度」（高中課本寫成 $|a|$），正式名稱是**範數 norm**。2D 就是畢氏定理：$(3, 4)$ 的長度是 $\sqrt{9 + 16} = 5$；3D 也一樣：$(1, 2, 2)$ 的長度是 $\sqrt{1 + 4 + 4} = 3$。

長度是 1 的向量叫**單位向量 unit vector**。向量除以自己的長度 $a / \lVert a \rVert$，方向不變、長度變成 1，只留下「方向」。每個數字都是 0 的**零向量 zero vector** 例外：長度是 0，不能除，也沒有方向。

In [ ]:
a = np.array([3.0, 4.0])
print(np.sqrt(np.sum(a ** 2)), np.linalg.norm(a))
u = a / np.linalg.norm(a)                           # 單位向量
print(u, np.linalg.norm(u))
print(len(a))

第一個 `print` 先照公式算，再用 NumPy 現成的 `np.linalg.norm`（`linalg` 是**線性代數 linear algebra**，研究向量與矩陣的數學）。

> **注意**：這個系列說向量的「長度」只指 $\lVert a \rVert$；有幾個數字，說「維度」。EP00、EP01 說的「長度 896 的向量」，從這集起改說「896 維向量」。最後一行的 `len(a)` 是數字個數 2，不是長度 5。

896 維畫不出來，但公式一模一樣：

In [ ]:
rng = np.random.default_rng(0)
h = rng.normal(size=896)            # 896 個亂數，假裝是一個 token 的向量
print(h.shape, np.linalg.norm(h))

這就是這集的策略：**在 2D 畫圖建立直覺，公式直接搬到 896 維。** EP00 ③ 的嵌入，就是把每個 token 換成這樣一個 896 維向量，只是數字是訓練學出來的，不是亂數。

## 2. 工具二：內積與餘弦相似度

### 對應位置相乘，再全部加起來

$$
a \cdot b = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n
$$

這叫**內積 dot product**，$a \cdot b$ 念「a dot b」。兩個維度相同的向量進去，**一個數**出來，例如 $(1, 2, 3) \cdot (4, 5, 6) = 4 + 10 + 18 = 32$。照 EP02 的習慣，先把 $\sum$ 寫成 for 迴圈，再換成 NumPy：

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
total = 0.0
for i in range(len(a)):
    total = total + a[i] * b[i]
print(total, np.sum(a * b), np.dot(a, b), a @ b)

`np.sum(a * b)` 是公式的字面翻譯，`np.dot(a, b)` 和 `a @ b` 是 NumPy 現成的內積。向量和自己的內積，就是長度的平方：$a \cdot a = \sum_i a_i^2 = \lVert a \rVert^2$。

### 幾何意義：長度 × 影子

$$
a \cdot b = \lVert a \rVert \lVert b \rVert \cos\theta
$$

$\theta$（念「theta」）是兩支箭頭的夾角。**餘弦 cosine** $\cos\theta$ 只跟角度有關：長度 1 的箭頭從 x 軸的正方向逆時針轉 $\theta$，它的 x 座標就是 $\cos\theta$，y 座標叫 $\sin\theta$（**正弦 sine**）。x 座標可以想成箭頭落在 x 軸上的「影子」：$\cos 0° = 1$、$\cos 60° = 0.5$、$\cos 90° = 0$、$\cos 180° = -1$。

把 $a$ 的方向當成 x 軸來看：箭頭 $b$ 的長度是 $\lVert b \rVert$，影子也跟著放大 $\lVert b \rVert$ 倍，所以 $\lVert b \rVert \cos\theta$ 就是 $b$ 落在 $a$ 的方向上的影子長（這個影子，高中課本叫 $b$ 在 $a$ 上的正射影），內積就是「**$a$ 的長度 × $b$ 在 $a$ 方向上的影子**」。夾角超過 90° 時，影子落在反方向，算成負的。

**最簡單的情況**：$a = (3, 0)$ 躺在 x 軸上，$b = (2, 2)$。$a \cdot b = 3 \times 2 + 0 \times 2 = 6$：$a$ 的第二個數字是 0，第二項消失了；剩下的 2 是 $b$ 的 x 座標，也就是 $b$ 落在 x 軸上的影子長。所以 6 ＝ 長度 3 × 影子 2。

In [ ]:
a = np.array([3.0, 0.0])
b = np.array([2.0, 2.0])     # 在斜線 y = x 上，和 a 夾 45°
fig, ax = plt.subplots(figsize=(4, 3))
for v, name in [(a, "a"), (b, "b")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>"))   # 從原點畫箭頭到 v
    ax.text(v[0] + 0.1, v[1] + 0.1, name)
ax.plot([2, 2], [0, 2], "k:")             # 從 b 的尖端垂直落到 x 軸（黑色點線）
ax.plot([0, 2], [-0.15, -0.15], lw=4)     # b 的影子，長 2
ax.text(0.6, -0.55, "shadow = 2")
ax.set(xlim=(-0.5, 3.8), ylim=(-0.8, 2.6), aspect="equal")
ax.grid(True)
plt.show()

$b$ 在斜線 $y = x$ 上，和 $a$ 夾 45°，用兩個公式算算看：

In [ ]:
a = np.array([3.0, 0.0])
b = np.array([2.0, 2.0])
print(a @ b, np.linalg.norm(a) * np.linalg.norm(b) * np.cos(np.radians(45)))

`np.cos` 吃的角度單位是**弧度 radian**（180° 等於 $\pi$ 弧度），`np.radians(45)` 把 45° 換成弧度。兩個公式都得到 6，最後一位的 `1` 是 EP02 看過的捨入誤差。

### 為什麼 Σ 會等於「長度 × 影子」

$a$ 不躺在 x 軸上也成立，證明要用高中的**餘弦定理 law of cosines**。$a$、$b$ 和連接兩個尖端的 $a - b$ 圍成一個三角形：從 $b$ 的尖端走 $a - b$ 就到 $a$ 的尖端（因為 $b + (a - b) = a$）。下圖用 $a = (3, 1)$、$b = (1, 2)$：

In [ ]:
from matplotlib.patches import Arc

a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])     # 和 a 夾 45°
fig, ax = plt.subplots(figsize=(4.4, 3.2))
for v, name in [(a, "a"), (b, "b")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", linewidth=1.5))
    ax.text(v[0] + 0.1, v[1] + 0.05, name)
# 從 b 的尖端走 a - b，就到 a 的尖端
ax.annotate("", xy=a, xytext=b, arrowprops=dict(arrowstyle="-|>", color="C3", linestyle="--", linewidth=1.5))
ax.text(2.1, 1.65, "a - b = (2, -1)", color="C3")
# 夾角 θ：從 a 的方向轉到 b 的方向
deg_a, deg_b = np.degrees(np.arctan2(a[1], a[0])), np.degrees(np.arctan2(b[1], b[0]))
ax.add_patch(Arc((0, 0), 1.2, 1.2, theta1=deg_a, theta2=deg_b, color="C0"))
ax.text(0.55, 0.5, "θ = 45°", color="C0")
ax.set(xlim=(-0.4, 3.9), ylim=(-0.4, 2.5), aspect="equal")
ax.grid(True)
plt.show()

餘弦定理（知道三角形的兩邊和夾角，就能算出第三邊）：

$$
\lVert a - b \rVert^2 = \lVert a \rVert^2 + \lVert b \rVert^2 - 2 \lVert a \rVert \lVert b \rVert \cos\theta
$$

左邊換成 $\sum$ 展開，每一項用乘法公式 $(x - y)^2 = x^2 + y^2 - 2xy$：

$$
\lVert a - b \rVert^2 = \sum_i (a_i - b_i)^2 = \sum_i a_i^2 + \sum_i b_i^2 - 2 \sum_i a_i b_i
$$

兩條式子算的是同一個 $\lVert a - b \rVert^2$，扣掉共同的 $\lVert a \rVert^2 + \lVert b \rVert^2$、再除以 $-2$，就得到 $\sum_i a_i b_i = \lVert a \rVert \lVert b \rVert \cos\theta$。用圖上的數字檢查第二條：左邊 $\lVert (2, -1) \rVert^2 = 4 + 1 = 5$，右邊 $10 + 5 - 2 \times 5 = 5$（$a \cdot b = 3 + 2 = 5$）。

反過來，有了內積，就能算出夾角：

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])
cos_theta = (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(cos_theta, np.degrees(np.arccos(cos_theta)))

`np.arccos` 是 cos 的反函數（EP02）：在 0° 到 180° 之間，cos 一路往下，不同的角度給出不同的值，所以給一個 −1 到 1 之間的數，就能算回唯一的角度（單位是弧度，`np.degrees` 換成度）。所以 $(3, 1)$ 和 $(1, 2)$ 夾 45°。

### 正、零、負：正負號就是方向

長度都是正的（零向量除外），所以內積的正負號由 $\cos\theta$ 決定：

| 夾角 | 內積 | 意思 |
|---|---|---|
| 小於 90° | 正 | 方向接近 |
| 剛好 90° | 0 | **垂直**，線性代數的正式說法是**正交 orthogonal** |
| 大於 90° | 負 | 方向偏向相反（180° 時完全相反） |

**先猜猜看**：下面三個向量和 $a = (2, 1)$ 的夾角，分別小於、剛好、大於 90°，內積各是正、零還是負？

In [ ]:
a = np.array([2.0, 1.0])
print(a @ np.array([1.0, 1.0]), a @ np.array([-1.0, 2.0]), a @ np.array([-2.0, -0.5]))

### 餘弦相似度：只看方向

$$
\text{cos\_sim}(a, b) = \frac{a \cdot b}{\lVert a \rVert \lVert b \rVert} = \cos\theta
$$

這叫**餘弦相似度 cosine similarity**，範圍是 $-1$ 到 $1$：同方向是 1、垂直是 0、反方向是 $-1$，和長度無關。零向量沒有方向，分母是 0，不能算（NumPy 會得到 `nan`）。

**為什麼要除以長度？** 把一個向量放大 100 倍，內積也跟著變 100 倍，但方向根本沒變。比較兩個東西「像不像」，應該只看方向。（網站上有可以拖動兩個 2D 向量的互動元件。）

下面用**手作的示範向量**算一張相似度表。4 個數字代表「動物、毛茸茸、甜、黃色」的程度，是我們自己定的，不是模型學出來的。`E` 是把 5 個向量一列一列疊起來的表格（矩陣，工具三介紹），`E[0]` 是貓那一列。`np.round(sim, 2)` 四捨五入到小數點後 2 位：

In [ ]:
def cos_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

#              動物 毛茸茸  甜  黃色
E = np.array([[1.0, 0.9, 0.0, 0.1],    # 貓
              [1.0, 0.6, 0.0, 0.3],    # 狗
              [1.0, 0.7, 0.0, 0.9],    # 小雞
              [0.0, 0.0, 0.8, 0.3],    # 蘋果
              [0.0, 0.0, 0.9, 1.0]])   # 香蕉
sim = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        sim[i, j] = cos_sim(E[i], E[j])
print(np.round(sim, 2))
# 貓和狗的內積，以及貓和「放大 100 倍的狗」的內積
print(E[0] @ E[1], E[0] @ (100 * E[1]))
print(cos_sim(E[0], E[1]), cos_sim(E[0], 100 * E[1]))

- 第 $i$ 列第 $j$ 欄是「第 $i$ 個詞和第 $j$ 個詞有多像」：對角線都是 1（自己和自己同方向），而且沿著對角線對稱。
- 貓和狗最像（0.97）；小雞因為「黃色」，和香蕉也有一點像（0.44）；動物和水果幾乎不像。
- 狗放大 100 倍，內積從 1.57 變成 157.0，相似度完全不變。
- 這裡的數字都不是負的，所以沒有負的相似度；真實的嵌入有正有負，相似度也可以是負的。

**真實的嵌入**：那 896 個數字不是人標的「動物、甜」，而是訓練學出來的（EP18），單獨看一個數字通常說不出它代表什麼。EP00 說「意思相近的 token，向量也相近」，「相近」最常用的量法就是餘弦相似度。

**和 LLM 的連結**：EP00 ⑤ 的分數也是內積。每個候選 token 有一個 896 維向量，最後一個位置的向量 $h$ 和它做內積，就是這個 token 的分數。（打分數直接用內積、不除以長度，原因在本集重點。）

## 3. 工具三：矩陣與矩陣乘法

### 形狀 (m, n)：m 列 n 欄

**矩陣 matrix** 是排成長方形的數字表格。形狀 `(m, n)` 代表 m **列 row**（橫的一排）、n **欄 column**（直的一排）。

> **注意**：這個系列 row 一律叫「列」、column 一律叫「欄」，不用「行」（中國大陸的「行」「列」用法和台灣剛好相反）。

$A_{ij}$ 是第 $i$ 列第 $j$ 欄的數字，在 Python 是 `A[i - 1, j - 1]`。向量一列一列疊起來就是矩陣：「今天天氣很」4 個 token 的向量疊起來，就是形狀 `(4, 896)` 的矩陣，每一列是一個 token。

### 矩陣乘向量：列觀點和欄觀點

形狀 `(m, n)` 的矩陣乘上 `(n,)` 的向量 $x$，得到 `(m,)` 的向量。數學上寫成 $Ax$，程式裡是 `A @ x`。有兩種看法：

- **列觀點 row view**：輸出的第 $i$ 個數字＝第 $i$ 列和 $x$ 的內積。m 列，就是 m 個內積。
- **欄觀點 column view**：輸出＝$x_1 \times$ 第 1 欄 $+ \cdots + x_n \times$ 第 n 欄，也就是各欄的**加權和 weighted sum**：每一欄先乘上自己的權重（$x$ 的數字），再加起來。

$$
\begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}
\begin{pmatrix} 2 \\ 1 \\ -1 \end{pmatrix}
= \begin{pmatrix} 2 + 2 - 3 \\ 8 + 5 - 6 \end{pmatrix}
= 2 \begin{pmatrix} 1 \\ 4 \end{pmatrix} + 1 \begin{pmatrix} 2 \\ 5 \end{pmatrix} - 1 \begin{pmatrix} 3 \\ 6 \end{pmatrix}
= \begin{pmatrix} 1 \\ 7 \end{pmatrix}
$$

第一個等號後面是列觀點，第二個等號後面是欄觀點。數學習慣把 $x$ 寫成直的一欄；NumPy 的一維陣列 `(3,)` 沒有直、橫之分，`A @ x` 會把它當成直的來算，結果也是一維的 `(2,)`。

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
print(A.shape, A[1, 2], A[1], A[:, 2])
x = np.array([2.0, 1.0, -1.0])
print(A @ x)
# 列觀點：每一列和 x 做內積
print(np.array([A[0] @ x, A[1] @ x]))
# 欄觀點：各欄的加權和
print(x[0] * A[:, 0] + x[1] * A[:, 1] + x[2] * A[:, 2])

`A[1]` 是第 2 列、`A[:, 2]` 是第 3 欄（Python 從 0 數，EP01 的切片）。**列觀點用來看「打分數」，欄觀點用來看「變換」**，等一下都會用到。

### 矩陣乘矩陣：每一格都是一個內積

形狀 `(m, n)` 的 $A$ 乘上 `(n, p)` 的 $B$，得到 `(m, p)` 的 $C$：

$$
C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}
$$

白話：$C$ 的第 $i$ 列第 $j$ 欄，是 $A$ 的第 $i$ 列和 $B$ 的第 $j$ 欄的內積。所以輸出的 m × p 格，**每一格都是一個內積**，這就是 EP01 形狀規則的由來：$A$ 的一列和 $B$ 的一欄都要有 n 個數字，內積才算得了。每一格要做 n 次乘法，總共是 **m × n × p 次乘法**。

In [ ]:
B = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, -1.0]])
C = A @ B                           # (2, 3) @ (3, 2) → (2, 2)
print(C)
# C[1, 0]＝A 的第 2 列和 B 的第 1 欄的內積（Python 從 0 數）
print(A[1] @ B[:, 0])
# 轉置：列和欄互換，(2, 3) 變 (3, 2)
print(A.T)

這裡一共 2 × 3 × 2 = 12 次乘法。`A.T` 是 EP01 看過的轉置。（網站上有逐格步進的互動元件：每一格亮起時，$A$ 對應的一列和 $B$ 對應的一欄一起亮。）

前面那張 5 × 5 的相似度表，也可以一次用矩陣乘法算完：每一列先變成單位向量，再乘上自己的轉置，25 個內積一起算：

In [ ]:
U = E / np.linalg.norm(E, axis=1, keepdims=True)              # 每一列除以自己的長度
print(U.shape, (U @ U.T).shape, np.allclose(U @ U.T, sim))

`keepdims=True` 讓每一列的長度排成 `(5, 1)`，靠廣播讓每一列除以**自己的**長度（EP01 Softmax 的老招）。`U @ U.T` 的第 $i$ 列第 $j$ 欄是 `U[i]` 和 `U[j]` 的內積，而兩個單位向量的內積，就是餘弦相似度。

### `@` 和 `*`、轉置、順序

- **`*` 是逐元素相乘**，兩邊形狀要一樣或能廣播（EP01）；**`@` 是矩陣乘法**，左邊的欄數要等於右邊的列數。兩者完全不同（弄壞它 1、2）。
- **轉置** `.T` 寫成數學是 $A^\top$，念「A 的轉置」。一維陣列沒有列和欄可以交換，`.T` 什麼都不做（弄壞它 3）。
- **$AB$ 通常不等於 $BA$**：數字相乘可以交換順序（$2 \times 3 = 3 \times 2$），矩陣一般不行。

In [ ]:
M = np.array([[1, 2], [3, 4]])
P = np.array([[0, 1], [1, 0]])
print(M @ P)
print(P @ M)

`M @ P` 把 `M` 的兩欄對調，`P @ M` 把兩列對調，結果不一樣。形狀不同時更明顯：`(2, 3) @ (3, 4)` 能算，倒過來根本不能算。

### LLM 怎麼替 151,936 個 token 打分數

詞彙表裡每個 token 都有一個用來打分數的 896 維向量，151,936 個疊起來，就是 EP00 說的反嵌入矩陣 $W$，形狀 `(151936, 896)`。最後一個位置的向量 $h$ 形狀是 `(896,)`。用列觀點看 `W @ h`：$W$ 的每一列和 $h$ 各做一次內積，**一次得到 151,936 個分數**，再交給 Softmax 換成機率。寫成 `h @ W.T`，也就是 `(896,) @ (896, 151936)`，是同一個計算。

In [ ]:
print(896 * 151_936, 896 * 151_936 / 5e8)

- 一次打分數要 **136,134,656 次乘法，約 1.36 億次**，而且每產生一個 token 就要做一次。這種計算量怎麼估算，留給 EP33。
- 第二個數字和 ③ 有關：嵌入表也是一張 `(151936, 896)` 的表，裡面有 896 × 151,936 個數字，每一個都是參數。`5e8` 是 $5 \times 10^8$，也就是 Qwen2.5-0.5B 約 5 億個參數；所以嵌入表就占了約 27%，這就是 EP00 說「其餘幾乎都是嵌入表」的原因。

**有興趣的話**：這份 notebook 的最後一格是選做程式，載入 Qwen2.5-0.5B，用上面 `U @ U.T` 的方法算幾個詞的真實嵌入相似度表。

## 4. 工具四：線性變換

### 矩陣是一台函數機器

$f(x) = Ax$：向量進去、向量出來，就是 EP02 的函數機器，`(m, n)` 的矩陣吃 n 維、吐 m 維。用矩陣這樣變換向量，叫**線性變換 linear transformation**。

### 看 (1, 0) 和 (0, 1) 被送到哪裡

記 $e_1 = (1, 0)$、$e_2 = (0, 1)$（這個 e 只是名字，和 EP02 的自然常數 $e$ 無關）。用欄觀點，$A e_1$ 是 1 × 第 1 欄 + 0 × 第 2 欄，也就是第 1 欄；$A e_2$ 就是第 2 欄。換句話說：

**矩陣的兩欄，就是 $(1, 0)$ 和 $(0, 1)$ 被送到的落點。**

任何向量都能拆成 $x = x_1 e_1 + x_2 e_2$，再用一次欄觀點：

$$
A x = x_1 (A e_1) + x_2 (A e_2)
$$

白話：$x$ 被送到的位置，是兩個落點的加權和，權重就是 $x$ 自己的兩個數字。例如下面的剪切把 $e_1$ 留在原地、把 $e_2$ 送到 $(1, 1)$，那 $(3, 2)$ 就被送到 $3(1, 0) + 2(1, 1) = (5, 2)$。

### 四個 2D 例子

| 變換 | 矩陣 | 白話 |
|---|---|---|
| 縮放 scaling | $\begin{pmatrix} 2 & 0 \\ 0 & 0.5 \end{pmatrix}$ | 橫向拉長 2 倍、縱向壓成一半 |
| 旋轉 rotation | $\begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ | 繞原點逆時針轉 90° |
| 剪切 shear（高中課本叫推移） | $\begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$ | 越高的點往右推得越多 |
| 投影 projection | $\begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}$ | 壓扁到 x 軸上，只留下影子 |

In [ ]:
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
scale = np.array([[2.0, 0.0], [0.0, 0.5]])
rot = np.array([[0.0, -1.0], [1.0, 0.0]])     # 逆時針轉 90°
shear = np.array([[1.0, 1.0], [0.0, 1.0]])
proj = np.array([[1.0, 0.0], [0.0, 0.0]])     # 投影到 x 軸
for M in [scale, rot, shear, proj]:
    print(M @ e1, M @ e2, M @ np.array([3.0, 2.0]))

- 每次印出的前兩個向量，剛好是矩陣的兩欄；第三個是 $(3, 2)$ 的落點，剪切的 $(5, 2)$ 和手算的一樣。
- 轉任意角度 $\theta$ 的矩陣是 $\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$：它把 $e_1$ 送到**單位圓 unit circle**（圓心在原點、半徑 1 的圓）上角度 $\theta$ 的點 $(\cos\theta, \sin\theta)$；代入 $\theta = 90°$ 就是 `rot`。EP24 的旋轉位置編碼 RoPE 會用到旋轉矩陣。
- 什麼都不做的矩陣叫**恆等矩陣 identity matrix** $I$（高中課本叫單位方陣）：對角線是 1、其他是 0，NumPy 是 `np.eye(2)`。$Ix = x$、$IA = AI = A$，就像乘以 1。

第 6 節會寫一個 `draw` 函式，畫出格線和**單位正方形 unit square**（四個角是 $(0, 0)$、$(1, 0)$、$(1, 1)$、$(0, 1)$）被矩陣變成什麼樣子。（網站上有可以拖動兩欄的互動元件。）

### 「線性」是什麼意思

$$
f(a + b) = f(a) + f(b), \qquad f(c\,a) = c\,f(a)
$$

白話：先相加再變換＝先變換再相加；先伸縮再變換＝先變換再伸縮（$c$ 是任何一個純量）。所有矩陣都符合這兩條。畫成圖，就是三件事：**直線還是直線、原點不動、格線保持平行而且等距**。

In [ ]:
rng = np.random.default_rng(0)
a, b = rng.normal(size=2), rng.normal(size=2)
print(np.allclose(shear @ (a + b), shear @ a + shear @ b))
print(np.allclose(shear @ (3 * a), 3 * (shear @ a)))

def shift(v):                       # 平移：每個點往右移 1
    return v + np.array([1.0, 0.0])

print(shift(np.zeros(2)), np.allclose(shift(a + b), shift(a) + shift(b)))

隨便挑兩個向量，剪切都符合。**平移**就不是線性的：它把原點移走了，也不符合第一條。逐元素平方也不是：它會把直線彎成曲線。

### 合成＝矩陣相乘，順序很重要

像 EP02 的合成函數 $f(g(x))$，「先旋轉 $R$、再剪切 $S$」寫成 $S(Rx)$。而 $S(Rx) = (SR)x$：兩步可以先合成一個矩陣 $SR$。**最靠近 $x$ 的矩陣先作用，所以要從右往左讀。**

In [ ]:
x = np.array([1.0, 2.0])
# 一步一步來：先旋轉，再剪切
print(rot @ x, shear @ (rot @ x))
# 先合成一個矩陣，再乘一次
SR = shear @ rot
print(SR)
print(SR @ x)

兩種做法都把 $(1, 2)$ 送到 $(-1, 1)$。順序一般不能交換：「先旋轉再剪切」和「先剪切再旋轉」是兩回事（弄壞它 4）。另外，只有線性變換的話，疊再多層都能先合成一個矩陣，等於只有一層；所以神經網路的層和層之間要夾著非線性的**激活函數 activation function**，EP10 會看到。

**能不能復原？** 投影把 $(3, 2)$ 和 $(3, -5)$ 都壓成 $(3, 0)$，無法知道原本是哪一個，就像 EP02 的 $x^2$ 沒有反函數；旋轉則可以轉回來。能復原的矩陣有**反矩陣 inverse matrix** $A^{-1}$（高中課本叫反方陣），暫時用不到。

### 神經網路的一層：`nn.Linear`

PyTorch 的 `nn.Linear(in, out)` 是神經網路最常見的一層，它算的是 `y = x @ W.T + b`：

- `W` 的形狀是 `(out, in)`，每一列負責一個輸出。只有一個向量時，`x @ W.T` 就是 $Wx$。`x` 也可以一次放很多個 token（每列一個），全部被同一個 `W` 變換。
- `b` 叫**偏差 bias**，靠廣播加到每一列。輸入 0 會得到 $b$，也就是原點被送到 $b$，嚴格說是**仿射變換 affine transformation**（線性變換再平移），但大家仍然叫它**線性層 linear layer**。

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(4, 3))         # 4 個 token，每個 3 維
W = rng.normal(size=(2, 3))         # PyTorch 的存法：(輸出, 輸入)
bias = np.array([0.5, -0.5])
Y = X @ W.T + bias                  # (4, 3) @ (3, 2) → (4, 2)，再廣播加上 (2,)
print(Y.shape, np.allclose(Y[0], W @ X[0] + bias))

第二個檢查：第一個 token 的輸出 `Y[0]`，就是 $W$ 乘上它的向量再加 $b$。Qwen2.5-0.5B 每一層的注意力和前饋網路，都由好幾個這樣的線性層組成，**每一層最主要的計算就是矩陣乘法**。`W` 裡的數字怎麼訓練出來，要等 EP04 的梯度和 EP05 的鏈鎖律。

## 5. 本集重點：內積＝相似度，矩陣乘法＝一次算很多個內積

### 一個查詢，對一排向量打分數

把 4 個 3 維向量疊成矩陣 `K`（形狀 `(4, 3)`），再拿一個**查詢 query** 向量 `q` 去問：「你們誰和我最像？」（「查詢」這個名字先借用，EP22 會正式介紹。）**先猜猜看**：哪一列拿到的權重最大？

In [ ]:
K = np.array([[2.0, 0.0, 1.0],      # 和 q 方向接近
              [0.0, 1.0, 0.0],      # 和 q 垂直
              [1.0, 1.0, 1.0],      # 有點像
              [-1.0, 0.0, -1.0]])   # 方向相反
q = np.array([1.0, 0.0, 1.0])
scores = K @ q                      # (4, 3) @ (3,) → (4,)：一次 4 個內積
weights = softmax(scores)
print(scores)
print(weights, weights.sum())

**內積 → 分數 → Softmax → 權重**：最像的 `K[0]` 拿到約 70% 的權重，垂直的 `K[1]` 只有 3.5%，方向相反的 `K[3]` 只剩 0.5%。這和 LLM 打分數是同一個骨架，只是 4 列變成 151,936 列、3 維變成 896 維。

> **注意**：這裡的「權重」是當場算出來、加起來是 1 的比例，不是 EP00 說的「權重＝模型的參數」（像 `nn.Linear` 的 `W`）。

### 長向量占便宜

這裡的「像」是**內積**，不是餘弦相似度：方向越接近，分數越高；但向量越長，分數也越大。把 `K[2]` 拉長 3 倍，方向不變：

In [ ]:
K_long = K.copy()
K_long[2] = [3.0, 3.0, 3.0]         # 和 K[2] 同方向、長 3 倍
print(K_long @ q, cos_sim(K_long[2], q), cos_sim(K[0], q))
# q 變 2 倍
print(np.round(softmax(K @ (2 * q)), 3))

`K_long[2]` 的方向沒變，和 `q` 的餘弦相似度是 0.82，輸給 `K[0]` 的 0.95；可是它的分數變成 6，跑到第一名。

那為什麼打分數不除以長度？比較兩個詞像不像時要除；但 LLM 打分數、注意力都直接用內積，因為**長度也是模型學出來、有用的資訊**。看第二行：`q` 變 2 倍，所有分數一起變 2 倍，Softmax 就更集中（最像的從 70% 變成 88%），等於把溫度從 1 調成 0.5。所以「內積＝相似度」要記成：**長度一樣時，方向越接近，分數越高；長度會把分數放大。**（網站上有可以拖動查詢向量、切換內積與餘弦相似度的互動元件。）

### 很多個查詢：一次矩陣乘法

查詢不只一個時，把它們也疊成矩陣：

In [ ]:
queries = np.array([[1.0, 0.0, 1.0],
                    [0.0, 2.0, -1.0]])   # 2 個查詢，形狀 (2, 3)
scores = queries @ K.T                  # (2, 3) @ (3, 4) → (2, 4)：8 個內積
print(scores)
print(np.round(softmax(scores, axis=-1), 3))

`(2, 4)` 的第 $i$ 列第 $j$ 欄，就是「第 $i$ 個查詢和 `K` 的第 $j$ 列的內積」。Softmax 沿著 `axis=-1` 做，每個查詢各自分到一組加起來是 1 的權重（EP01）。第二個查詢 $(0, 2, -1)$ 和 `K[1]` 最像，拿到 56%。

**這就是注意力的骨架。** 每個 token 拿自己的查詢向量，和每個 token（包括自己）的向量做內積；4 個 token 一次算完，就是 `(4, d) @ (d, 4)` → `(4, 4)` 的分數表（$d$ 是向量的維度）。Softmax 之後得到權重，再用權重把那些 token 的資訊**加權平均 weighted average**（權重加起來是 1 的加權和）。實際的 LLM 還會讓每個 token 只看自己和前面的 token，也會把向量切成好幾份分開算，這些是 EP23 的事；查詢和被比較的向量從哪裡來、為什麼還要除以 $\sqrt{d}$，是 EP22 的事。

## 6. 動手做：從零寫出 5 個函式

在下一格寫完這 5 個函式（網站上是寫在 `llm-labs/ep03/linalg.py`），再執行下一格的檢查。

要求與提示：

1. **只用純 Python**（list、for、`math.sqrt`），不能用 NumPy。**巢狀 list nested list** 是「list 裡面放 list」，例如 `[[1, 2, 3], [4, 5, 6]]`，每個內層的 list 是一列，形狀是 `(len(A), len(A[0]))`。
2. **`norm`** 用你的 `dot(a, a)`，因為 $a \cdot a = \lVert a \rVert^2$。
3. **錯誤訊息要說出兩個形狀**。`raise ValueError("訊息")` 會讓程式停下來並顯示訊息。把 tuple 放進 f-string：`f"{(m, n)}"` 會印成 `(2, 3)`，`f"{(len(a),)}"` 會印成 `(3,)`。
4. **`matvec`** 用列觀點，呼叫 `dot`。**`matmul`** 用三層迴圈：`i` 跑 $A$ 的列、`j` 跑 $B$ 的欄、`k` 跑內積的每一項，就是工具三 $C_{ij}$ 的公式。

In [ ]:
import math


def dot(a, b):
    """a、b 是元素個數相同的 list，回傳 Σ a_i b_i（一個數）。個數不同要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 dot")


def norm(a):
    """回傳向量 a 的長度 ‖a‖ = √(Σ a_i²)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 norm")


def cosine_similarity(a, b):
    """回傳 a·b / (‖a‖‖b‖)。維度不同（list 元素個數不同）、或其中一個是零向量（沒有方向），raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 cosine_similarity")


def matvec(A, x):
    """A 是 m 列 n 欄的巢狀 list，x 是 n 個數字的 list，回傳 m 個數字的 list。
    形狀不合要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 matvec")


def matmul(A, B):
    """A 的形狀 (m, n)、B 的形狀 (n, p)，回傳 (m, p) 的巢狀 list。用三層迴圈。
    形狀不合要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 matmul")

下一格的 7 個檢查和網站上的 `tests/test_ep03_linalg.py` 一樣，依序檢查：`dot` 和 NumPy 一樣；`norm` 的 3、4、5（和 896 維）；餘弦相似度的同方向、垂直、反方向，以及零向量要報錯；`matvec`、`matmul` 和 NumPy 一樣，乘恆等矩陣不變；形狀不合時要報 `ValueError`，而且錯誤訊息說出兩個形狀；`matmul` 不能交換順序；最後和 PyTorch 的 `torch.matmul`、`cosine_similarity`、`nn.Linear` 比對（沒裝 PyTorch 就跳過）。幾個新寫法：

- `.tolist()` 把 NumPy 陣列換成（巢狀）list，因為你的函式只吃 list。
- `with raises(ValueError) as e:` 是「底下這段程式應該要報 `ValueError`」，沒報錯，檢查就失敗；`str(e.value)` 是錯誤訊息。（網站上的測試用 pytest 的 `pytest.raises`，這裡寫了一個用法一樣的小工具，notebook 就不用裝 pytest。）
- `math.isclose`（EP02）預設比相對誤差，和 0 比的時候，只要結果帶一點點捨入誤差（例如 `1e-17`）就不算相等，所以垂直那一行加上 `abs_tol=1e-12`：差距在 $10^{-12}$ 以內就算相等。
- `layer.weight`、`layer.bias` 就是這一層的 `W` 和 `b`。`.detach()` 拿掉 PyTorch 為了訓練記下的計算紀錄（EP09），才能用 `.numpy()` 轉成 NumPy 陣列；`.item()` 把只有一個數的張量取出成 Python 的數。

改了上一格之後，要重新執行上一格，再執行這一格。

In [ ]:
import contextlib


def check_linalg():
    """和網站上 tests/test_ep03_linalg.py 一樣的 7 個測試。"""

    class Skipped(Exception):
        pass

    class ExcInfo:
        value = None

    @contextlib.contextmanager
    def raises(exc_type):
        """和 pytest.raises 一樣：底下這段程式應該要報 exc_type，沒報錯就算失敗。"""
        info = ExcInfo()
        try:
            yield info
        except exc_type as e:
            info.value = e
        else:
            raise AssertionError(f"應該要報 {exc_type.__name__}，卻沒有報錯")

    def dot_matches_numpy():
        assert dot([1, 2, 3], [4, 5, 6]) == 32            # 4 + 10 + 18
        rng = np.random.default_rng(0)
        a, b = rng.normal(size=50), rng.normal(size=50)
        assert math.isclose(dot(a.tolist(), b.tolist()), np.dot(a, b))

    def norm_3_4_5():
        assert norm([3, 4]) == 5.0                        # 畢氏定理
        assert norm([1, 2, 2]) == 3.0
        a = np.random.default_rng(1).normal(size=896)
        assert math.isclose(norm(a.tolist()), np.linalg.norm(a))

    def cosine_parallel_orthogonal_opposite():
        assert math.isclose(cosine_similarity([1, 2], [3, 6]), 1.0)       # 同方向，長度不同
        assert math.isclose(cosine_similarity([1, 2], [-2, 1]), 0.0, abs_tol=1e-12)   # 垂直
        assert math.isclose(cosine_similarity([1, 2], [-1, -2]), -1.0)    # 反方向
        with raises(ValueError):
            cosine_similarity([0, 0], [1, 2])                             # 零向量沒有方向
        with raises(ValueError):
            cosine_similarity([1, 2], [0, 0])                             # 兩邊都要檢查

    def matvec_matmul_match_numpy():
        assert matvec([[1, 2, 3], [4, 5, 6]], [2, 1, -1]) == [1, 7]
        rng = np.random.default_rng(2)
        A, x, B = rng.normal(size=(3, 4)), rng.normal(size=4), rng.normal(size=(4, 5))
        assert np.allclose(matvec(A.tolist(), x.tolist()), A @ x)
        C = matmul(A.tolist(), B.tolist())
        assert np.array(C).shape == (3, 5) and np.allclose(C, A @ B)
        assert np.allclose(matmul(A.tolist(), np.eye(4).tolist()), A)    # 乘恆等矩陣不變

    def shape_mismatch_raises():
        with raises(ValueError) as e:
            dot([1, 2, 3], [1, 2])
        assert "(3,)" in str(e.value) and "(2,)" in str(e.value)
        with raises(ValueError) as e:
            matvec([[1, 2, 3], [4, 5, 6]], [1, 2])
        assert "(2, 3)" in str(e.value) and "(2,)" in str(e.value)
        with raises(ValueError) as e:
            matmul([[1, 2, 3], [4, 5, 6]], [[1, 2, 3], [4, 5, 6]])     # (2, 3) @ (2, 3)
        assert str(e.value).count("(2, 3)") == 2

    def matmul_is_not_commutative():
        A = [[1, 2], [3, 4]]
        P = [[0, 1], [1, 0]]
        assert matmul(A, P) == [[2, 1], [4, 3]]           # A 的兩欄對調
        assert matmul(P, A) == [[3, 4], [1, 2]]           # A 的兩列對調

    def matches_pytorch():
        try:
            import torch
        except ImportError:
            raise Skipped("沒有安裝 PyTorch")
        rng = np.random.default_rng(4)
        A, B = rng.normal(size=(3, 4)), rng.normal(size=(4, 5))
        ours = matmul(A.tolist(), B.tolist())
        assert np.allclose(ours, torch.matmul(torch.from_numpy(A), torch.from_numpy(B)).numpy())
        a, b = rng.normal(size=8), rng.normal(size=8)
        theirs = torch.nn.functional.cosine_similarity(torch.from_numpy(a), torch.from_numpy(b), dim=0)
        assert math.isclose(cosine_similarity(a.tolist(), b.tolist()), theirs.item(), rel_tol=1e-6)
        torch.manual_seed(0)
        layer, x = torch.nn.Linear(3, 2), torch.randn(4, 3)        # 3 進 2 出；4 個 token
        W, bias = layer.weight.detach().numpy(), layer.bias.detach().numpy()
        assert W.shape == (2, 3)                                    # (輸出, 輸入)
        assert np.allclose(layer(x).detach().numpy(), x.numpy() @ W.T + bias, atol=1e-6)

    tests = [dot_matches_numpy, norm_3_4_5, cosine_parallel_orthogonal_opposite, matvec_matmul_match_numpy,
             shape_mismatch_raises, matmul_is_not_commutative, matches_pytorch]
    passed = skipped = pending = 0
    for t in tests:
        try:
            t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError as e:
            print("…", t.__name__, f"（{e}）")
            pending += 1
        except Skipped as e:
            print("-", t.__name__, f"跳過（{e}）")
            skipped += 1
        except Exception as e:
            print("✗", t.__name__, type(e).__name__, e)
    summary = f"{passed}/{len(tests)} 通過"
    if skipped:
        summary += f"，{skipped} 個跳過"
    if pending:
        summary += f"，{pending} 個還沒寫好：先完成上一格，或執行最後的參考解答"
    print(summary)


check_linalg()

### 畫出變形圖：`draw`

網站上這是一個畫圖腳本 `llm-labs/ep03/plot_transform.py`，執行後存成 `ep03/transform.png`；在 notebook 裡只要在下一格寫好 `draw` 這個函式，再下一格會用它畫圖、直接顯示。

`draw(ax, M, title)` 畫三樣東西，再用 `ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", title=title)` 把兩軸都設成 $-3$ 到 $3$、比例一樣，加上標題（EP02 用過的 `ax.set`）：

- **格線**：$x = -2, \ldots, 2$ 和 $y = -2, \ldots, 2$ 共 10 條線。每條取 101 個點排成 `(2, 101)`（`line[0]` 是 x 座標、`line[1]` 是 y 座標），一欄是一個點。`M @ line` 的第 $j$ 欄只用到 `line` 的第 $j$ 欄（$C_{ij}$ 的公式），就是 `M` 乘上第 $j$ 個點，一次變換整條線。原本的畫淡灰色，變換後的畫彩色。（`nn.Linear` 是一列一個 token，所以寫成 `X @ W.T`。）
- **單位正方形**：4 個角排成 `(2, 4)`，變換後用 `ax.fill(xs, ys, alpha=0.3)` 塗色。
- **`M @ e1` 和 `M @ e2` 兩支箭頭**：`ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=c))`。箭頭進不了圖例，另外畫一條「空線」把落點寫進圖例：`ax.plot([], [], color=c, label=f"M @ e1 = ({x:.2f}, {y:.2f})")`，最後 `ax.legend()`。

In [ ]:
def draw(ax, M, title):
    """在 ax 上畫出 2 × 2 矩陣 M 把格線、單位正方形、e1 和 e2 送到哪裡。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw")

下一格用你的 `draw` 畫一列三張圖：恆等、剪切、逆時針旋轉 30°，再檢查圖例裡的落點。畫對的話，剪切的 $e_2$ 落在 $(1.00, 1.00)$；旋轉 30° 的 $e_1$ 落在 $(0.87, 0.50)$、$e_2$ 落在 $(-0.50, 0.87)$。

In [ ]:
def show_transforms():
    """一列三張圖：恆等、剪切、逆時針旋轉 30°，再檢查圖例裡 e1、e2 的落點。"""
    c, s = np.cos(np.radians(30)), np.sin(np.radians(30))
    panels = [
        (np.eye(2), "identity  [[1, 0], [0, 1]]"),
        (np.array([[1.0, 1.0], [0.0, 1.0]]), "shear  [[1, 1], [0, 1]]"),
        (np.array([[c, -s], [s, c]]), "rotate 30 degrees"),
    ]
    expected = [
        ["M @ e1 = (1.00, 0.00)", "M @ e2 = (0.00, 1.00)"],
        ["M @ e1 = (1.00, 0.00)", "M @ e2 = (1.00, 1.00)"],
        ["M @ e1 = (0.87, 0.50)", "M @ e2 = (-0.50, 0.87)"],
    ]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
    try:
        for ax, (M, title) in zip(axes, panels):
            draw(ax, M, title)
    except NotImplementedError as e:
        plt.close(fig)                  # 不顯示空白的圖
        print(f"{e}：先完成上一格的 draw，或執行最後的參考解答，再回來執行這格")
        return
    labels = [ax.get_legend_handles_labels()[1] for ax in axes]   # 每張圖的圖例文字
    plt.show()
    for (M, title), want, got in zip(panels, expected, labels):
        missing = [w for w in want if w not in got]
        if missing:
            print("✗", f"{title}：圖例裡沒有 {missing}，現在是 {got}")
        else:
            print("✓", f"{title}：" + "、".join(want))


show_transforms()

網站上的腳本把主程式放在 `if __name__ == "__main__":` 底下，用 `matplotlib.use("Agg")` 只存檔、不開視窗；在 notebook 裡用 `plt.show()` 直接顯示就好。弄壞它 4 還會再用到 `draw`。

### 和 NumPy 比速度

用你寫的 `matmul` 算兩個 200 × 200 的矩陣相乘，和 NumPy 的 `@` 比。**先猜猜看**：差幾倍？

In [ ]:
import time

rng = np.random.default_rng(0)
A, B = rng.normal(size=(200, 200)), rng.normal(size=(200, 200))
try:
    t0 = time.perf_counter()
    C = matmul(A.tolist(), B.tolist())  # 你在上面寫的 matmul
    t1 = time.perf_counter()
    for _ in range(100):                # A @ B 太快，跑 100 次取平均
        D = A @ B
    t2 = time.perf_counter()
    py, nps = t1 - t0, (t2 - t1) / 100
    print(np.allclose(C, D), f"{py:.3f} 秒", f"{nps * 1000:.3f} 毫秒", f"{py / nps:.0f} 倍")
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

800 萬次乘法，結果一樣，速度差了幾千倍。倍數每次都不一樣：NumPy 只要零點幾毫秒，一點干擾倍數就跳很多，可以多跑幾次；換一台電腦（例如 Colab）也會不一樣。這就是 EP01 的向量化：Python 迴圈一步一步算，NumPy 的 `@` 交給最佳化過的程式整批算，通常還會用上好幾個 CPU 核心。

## 7. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。完整的解釋在網站的「弄壞它」段落。

弄壞它 4 用到工具四的 `shear`、`rot` 和你在第 6 節寫的 `draw`；還沒寫好的話，那一格會提醒你，也可以先執行最後的參考解答，再回來執行。弄壞它 5 用到工具二的 `cos_sim`。

### 弄壞它 1：用 `*` 代替 `@`

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
# 想算內積，卻寫成 *
print(a * b)
# 換成直的 (3, 1) 和橫的 (1, 3)，還是用 *
col, row = a.reshape(3, 1), b.reshape(1, 3)
print((col * row).shape)

`*` 只做了內積的前半：對應位置相乘，卻沒有加起來。換成直的和橫的更糟：廣播把它延伸成「每個乘每個」的 3 × 3 表格，而且完全不報錯（EP01 弄壞它 1 的陷阱）。內積要用 `a @ b`；手上是橫的和直的，就用 `row @ col`，`(1, 3) @ (3, 1)` 得到 `(1, 1)`：

In [ ]:
print(col * row)
print(a @ b, row @ col)

### 弄壞它 2：形狀不合

讀懂錯誤訊息，想想要怎麼改。（這裡用 `try`／`except` 接住錯誤、只印出訊息，notebook 才不會停下來。）

In [ ]:
A = np.ones((2, 3))
B = np.ones((2, 3))
try:
    A @ B
except ValueError as e:
    print("ValueError:", e)

翻譯一下：operand 1 是第二個**運算元 operand**，也就是 `B`（從 0 數）。`(n?,k),(k,m?)->(n?,m?)` 是 `@` 的形狀規則：左邊 `(n, k)`、右邊 `(k, m)`、結果 `(n, m)`，兩個 `k` 必須一樣（問號代表那一維可以沒有，例如一維陣列）。`B` 的第 0 維（2 列）和 `A` 的最後一維（3 欄）對不上。

轉置其中一個就能算，但意思不同：

In [ ]:
print((A @ B.T).shape, (A.T @ B).shape)

`A @ B.T` 是兩邊的**列**兩兩做內積，`A.T @ B` 是兩邊的**欄**兩兩做內積。怎麼改要看你想讓「誰和誰」做內積，不是看哪個不報錯。

### 弄壞它 3：一維陣列轉置

`v.T` 會把 `v` 變成直的嗎？`v @ v.T` 是一個數，還是 3 × 3 的矩陣？

In [ ]:
v = np.array([1.0, 2.0, 3.0])
print(v.shape, v.T.shape)
print(v @ v.T)

一維陣列沒有列和欄可以交換，`.T` 什麼都不做，`v @ v.T` 就是內積 $1 + 4 + 9 = 14$。課本裡的 $v$ 常是直的一欄，$v v^\top$ 是 3 × 3 矩陣；在 NumPy 要先把 `v` 變成直的 `(3, 1)`，`.T` 才有作用：

In [ ]:
col = v.reshape(3, 1)
print(col.shape, col.T.shape, (col @ col.T).shape)

### 弄壞它 4：先旋轉再剪切 vs 先剪切再旋轉

先猜：兩種順序各把 $e_1$ 送到哪裡？再印出來、畫出來：

In [ ]:
# 先旋轉、再剪切
print(shear @ rot)
# 先剪切、再旋轉
print(rot @ shear)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
try:
    draw(axes[0], shear @ rot, "rotate, then shear")   # 你在第 6 節寫的 draw
    draw(axes[1], rot @ shear, "shear, then rotate")
    plt.show()
except NotImplementedError as e:
    plt.close(fig)                  # 不顯示空白的圖
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

看第一欄（$e_1$ 的落點）：先旋轉再剪切，$e_1$ 先轉到 $(0, 1)$，再被推到 $(1, 1)$；先剪切再旋轉，$e_1$ 剪切時不動，再轉到 $(0, 1)$。兩張圖的平行四邊形往相反方向傾斜：乘法的順序就是做事的順序，從右往左讀。

### 弄壞它 5：只看內積、不看長度

哪一個和 `q` 的內積比較大？餘弦相似度呢？

In [ ]:
q = np.array([1.0, 1.0])
same = np.array([1.0, 1.0])         # 方向和 q 完全一樣
big = np.array([10.0, 0.0])         # 長很多，但偏了 45°
print(q @ same, q @ big)
print(cos_sim(q, same), cos_sim(q, big))

內積說 `big` 比較像，只因為它長；餘弦相似度說 `same` 完全同方向（`...98` 是捨入誤差），`big` 偏了 45°（$\cos 45° \approx 0.707$）。所以比較兩個詞像不像要除以長度；打分數卻直接用內積，原因在本集重點。

EP22 會看到注意力把內積除以 $\sqrt{d}$（$d$ 是維度）：維度越高，加起來的項越多，內積的絕對值通常越大（各項有正有負會互相抵消，所以大約跟著 $\sqrt{d}$ 變大）。但 $\sqrt{d}$ 是固定的數，不像餘弦相似度那樣除掉每個向量自己的長度。

## 下一集

這集的矩陣都是固定的數字，可是模型的權重要靠訓練一點一點調整：每個權重該往哪個方向調、調多少？下一集 **EP04 導數與梯度：從斜率到梯度**，從 EP02 提過的斜率出發，學會導數、偏導數，以及指向「上升最快」方向的梯度。

## 參考解答

先自己寫，卡住再看。執行下一格會用參考解答取代第 6 節的 5 個函式，並重新跑一次檢查；再下一格是 `draw` 的參考解答，會重新畫出三張圖。之後可以回到上面，重新執行速度比較和弄壞它 4。

In [ ]:
# 只用純 Python（list 和 math），不用 NumPy。
import math


def _matrix_shape(A):
    """回傳巢狀 list A 的形狀 (m, n)；每一列的數字個數不同就 raise ValueError。"""
    m = len(A)
    n = len(A[0]) if m > 0 else 0
    for row in A:
        if len(row) != n:
            raise ValueError(f"每一列的數字個數要一樣，第 0 列有 {n} 個，有一列有 {len(row)} 個")
    return (m, n)


def dot(a, b):
    """a、b 是元素個數相同的 list，回傳 Σ a_i b_i（一個數）。個數不同要 raise ValueError。"""
    if len(a) != len(b):
        raise ValueError(f"dot：兩個向量的維度要一樣，現在形狀是 {(len(a),)} 和 {(len(b),)}")
    total = 0.0
    for i in range(len(a)):
        total = total + a[i] * b[i]
    return total


def norm(a):
    """回傳向量 a 的長度 ‖a‖ = √(Σ a_i²)。"""
    return math.sqrt(dot(a, a))


def cosine_similarity(a, b):
    """回傳 a·b / (‖a‖‖b‖)。維度不同（list 元素個數不同）、或其中一個是零向量（沒有方向），raise ValueError。"""
    na, nb = norm(a), norm(b)
    if na == 0.0 or nb == 0.0:
        raise ValueError("cosine_similarity：零向量沒有方向，餘弦相似度沒有定義")
    return dot(a, b) / (na * nb)


def matvec(A, x):
    """A 是 m 列 n 欄的巢狀 list，x 是 n 個數字的 list，回傳 m 個數字的 list。
    形狀不合要 raise ValueError。"""
    m, n = _matrix_shape(A)
    if n != len(x):
        raise ValueError(
            f"matvec：A 的形狀 {(m, n)} 和 x 的形狀 {(len(x),)} 對不上：A 的欄數要等於 x 的數字個數"
        )
    return [dot(row, x) for row in A]              # 列觀點：第 i 個＝第 i 列和 x 的內積


def matmul(A, B):
    """A 的形狀 (m, n)、B 的形狀 (n, p)，回傳 (m, p) 的巢狀 list。用三層迴圈。
    形狀不合要 raise ValueError。"""
    m, n = _matrix_shape(A)
    n2, p = _matrix_shape(B)
    if n != n2:
        raise ValueError(
            f"matmul：A 的形狀 {(m, n)} 和 B 的形狀 {(n2, p)} 對不上：A 的欄數 {n} 要等於 B 的列數 {n2}"
        )
    C = [[0.0] * p for _ in range(m)]
    for i in range(m):              # A 的第 i 列
        for j in range(p):          # B 的第 j 欄
            total = 0.0
            for k in range(n):      # 第 i 列和第 j 欄的內積
                total = total + A[i][k] * B[k][j]
            C[i][j] = total
    return C


check_linalg()

In [ ]:
def draw(ax, M, title):
    """在 ax 上畫出 2 × 2 矩陣 M 把格線、單位正方形、e1 和 e2 送到哪裡。"""
    M = np.asarray(M, dtype=float)
    t = np.linspace(-2, 2, 101)                       # 每條格線取 101 個點
    for c in range(-2, 3):
        for line in (np.stack([np.full_like(t, c), t]), np.stack([t, np.full_like(t, c)])):
            ax.plot(*line, color="lightgray", linewidth=0.8)          # 原本的格線
            ax.plot(*(M @ line), color="C0", linewidth=0.8, alpha=0.6)  # 變換後：整條線一起乘 M
    square = np.array([[0, 1, 1, 0], [0, 0, 1, 1]], dtype=float)     # 4 個角，一欄一個點
    ax.fill(*(M @ square), color="C2", alpha=0.3, label="unit square after M")
    for e, color, name in [((1, 0), "C3", "e1"), ((0, 1), "C4", "e2")]:
        x, y = M @ np.array(e, dtype=float)
        ax.annotate("", xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=color, linewidth=2))
        ax.plot([], [], color=color, linewidth=2, label=f"M @ {name} = ({x:.2f}, {y:.2f})")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", title=title)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.legend(loc="lower right", fontsize=8)


show_transforms()

## 選做：真實嵌入的相似度表（Qwen2.5-0.5B）

這格是選做，**不跑也不影響其他格**。它會從 Hugging Face 下載 EP00 用過的 Qwen2.5-0.5B（約 1 GB），用工具三 `U @ U.T` 的方法，算幾個詞（cat、dog、apple、banana、貓、狗、蘋果）的真實嵌入相似度表；只留剛好是 1 個 token 的詞。英文詞前面的空格是故意的：句子中間的英文詞，token 通常連著前面的空格。最後一行印出嵌入表的形狀、`tie_word_embeddings` 的設定（打分數的表是不是直接共用嵌入表），以及打分數那張表 `lm_head` 的形狀。

需要 PyTorch 和 transformers，Colab 都已經內建；Colab 免費版就能跑，用預設的 CPU 就夠，不用切換 GPU。下載和載入要等一下。這格預設不執行：把第一行改成 `RUN_QWEN = True`，再執行這格。結果請自己執行看看，再和工具二手作的相似度表比一比。

In [ ]:
RUN_QWEN = False   # 改成 True 再執行這格（會下載約 1 GB 的模型）

if RUN_QWEN:
    import torch.nn.functional as F
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
    emb = model.get_input_embeddings().weight.detach().float()   # 嵌入表
    cands = [" cat", " dog", " apple", " banana", "貓", "狗", "蘋果"]
    ids = {w: tok.encode(w, add_special_tokens=False) for w in cands}
    words = [w for w in cands if len(ids[w]) == 1]               # 只留剛好 1 個 token 的詞
    V = F.normalize(emb[[ids[w][0] for w in words]], dim=-1)     # 每一列變成單位向量（同工具三的 U）
    print(words)
    print((V @ V.T).numpy().round(2))
    print(emb.shape, model.config.tie_word_embeddings, model.lm_head.weight.shape)
else:
    print("RUN_QWEN = False：這格預設不執行。想跑的話，把第一行改成 RUN_QWEN = True，再執行這格。")